# Fasl · Notebook 2: Train the wheat models
Trains two image classifiers on the clean data from Notebook 1 (`fasl-clean`):
- **Health model** (8 classes): healthy, brown_rust, yellow_rust, stem_rust, septoria, loose_smut, powdery_mildew, **not_wheat**
- **Stage model** (4 classes): vegetative, flowering, grain_filling, ripening

Both use EfficientNet-B0 pretrained on ImageNet. At the end both are packed into **one ONNX file** for the API, and all real test numbers are written to `metrics.json` for the website.

**Settings before running:** Accelerator **GPU T4 x2** (or P100), Internet **On**, input dataset `fasl-clean` attached.

## Step 1: Setup
Imports, settings and the GPU check. Also finds the `fasl-clean` dataset automatically.

In [ ]:
import os, json, time, random, math, copy
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

INPUT_ROOT = Path(os.getenv("FASL_INPUT", "/kaggle/input"))
OUT = Path(os.getenv("FASL_OUT", "/kaggle/working/model"))
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
IMG = 224
BATCH = 32
WORKERS = 2
ARCH = "efficientnet_b0"
PRETRAINED = os.getenv("FASL_PRETRAINED", "1") == "1"
EPOCHS = {"health": int(os.getenv("FASL_EPOCHS_H", 15)), "stage": int(os.getenv("FASL_EPOCHS_S", 25))}
LR = 3e-4
LOW_QUALITY_P = 0.5     # share of training images turned into cheap-phone copies
PATIENCE = 8            # stop if val macro-F1 does not improve for this many epochs
MODEL_VERSION = "fasl-wheat-v0.5"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

import glob as _glob

def find_inputs(pattern):
    """Files under the input folder. glob.glob follows Kaggle's symlinked dataset folders;
    Path.rglob does not (Python 3.13), which made attached files invisible."""
    return sorted(Path(p) for p in _glob.glob(str(INPUT_ROOT / "**" / pattern), recursive=True))

manifests = find_inputs("manifest.csv")
assert manifests, "fasl-clean not attached: no manifest.csv under /kaggle/input"
DATA = manifests[0].parent
man = pd.read_csv(DATA / "manifest.csv")
print("Data folder:", DATA, "| images:", len(man))
print(pd.crosstab([man.task, man.label], man.split).to_string())

## Step 2: Build the two training tables
- Health model: all `health` images plus `not_wheat` (so it can reject non-wheat photos).
- Stage model: all `stage` images.

In [ ]:
HEALTH = ["healthy","brown_rust","yellow_rust","stem_rust","septoria","loose_smut","powdery_mildew","not_wheat"]
STAGE = ["vegetative","flowering","grain_filling","ripening"]
TASKS = {
    "health": (man[man.task.isin(["health","not_wheat"])].copy(), HEALTH),
    "stage":  (man[man.task == "stage"].copy(), STAGE),
}
for name, (d, labels) in TASKS.items():
    labels[:] = [l for l in labels if l in set(d.label)]     # drop classes with no images
    d["y"] = d.label.map({l: i for i, l in enumerate(labels)})
    print(name, "classes:", labels, "| train/val/test:", [int((d.split == s).sum()) for s in ["train","val","test"]])

## Step 3: Image loading and augmentation
- **Pad to square** first, so long thin leaf photos are not stretched. The API does exactly the same.
- Training images get random crops, flips, rotation and colour changes so the model copes with real phone photos.
- **Cheap-phone copies:** half the training images are shrunk (64 to 200 px), saved as low-quality JPEG and sometimes given noise, so the model still works for farmers with basic phones.
- `lowq_tf` makes a fixed low-quality copy of the test set for Step 7.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def pad_square(img):
    w, h = img.size; s = max(w, h)
    canvas = Image.new("RGB", (s, s), tuple(int(m * 255) for m in MEAN))
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas

import io as _io

def cheap_phone(img, rng=random):
    """Make a photo look like it came from a cheap phone: low resolution, heavy JPEG, some noise."""
    w, h = img.size
    short = rng.randint(64, 200)                                  # what a low-end camera or WhatsApp leaves
    s = short / min(w, h)
    if s < 1:
        img = img.resize((max(1, int(w * s)), max(1, int(h * s))), Image.BILINEAR)
    buf = _io.BytesIO(); img.save(buf, "JPEG", quality=rng.randint(15, 60))
    img = Image.open(buf).convert("RGB")
    if rng.random() < 0.3:                                        # sensor noise in dim light
        a = np.asarray(img, dtype=np.float32)
        a += np.random.normal(0, rng.uniform(3, 10), a.shape)
        img = Image.fromarray(a.clip(0, 255).astype(np.uint8))
    return img.resize((w, h), Image.BILINEAR)

def maybe_cheap_phone(img):
    return cheap_phone(img) if random.random() < LOW_QUALITY_P else img

train_tf = T.Compose([
    T.Lambda(maybe_cheap_phone),
    T.Lambda(pad_square),
    T.RandomResizedCrop(IMG, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(15),
    T.ColorJitter(0.3, 0.3, 0.2, 0.03),
    T.RandomApply([T.GaussianBlur(5)], p=0.15),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Lambda(pad_square), T.Resize((IMG, IMG)), T.ToTensor(), T.Normalize(MEAN, STD)])
# Same test images, but as a fixed cheap-phone copy (seeded), to measure how the model copes with poor photos.
lowq_tf = T.Compose([T.Lambda(lambda im: cheap_phone(im, random.Random(hash(im.size)))), eval_tf])

class ImgDS(Dataset):
    def __init__(self, d, tf): self.p, self.y, self.tf = list(d.path), list(d.y), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        img = Image.open(DATA / self.p[i]).convert("RGB")
        return self.tf(img), self.y[i]

def loaders(d):
    mk = lambda s, tf, sh: DataLoader(ImgDS(d[d.split == s], tf), batch_size=BATCH, shuffle=sh,
                                      num_workers=WORKERS, pin_memory=DEVICE == "cuda")
    return mk("train", train_tf, True), mk("val", eval_tf, False), mk("test", eval_tf, False)

# preview a few augmented training images
d, labels = TASKS["health"]
ds = ImgDS(d[d.split == "train"].sample(8, random_state=SEED), train_tf)
fig, ax = plt.subplots(1, 8, figsize=(16, 2.4))
for a, (x, y) in zip(ax, [ds[i] for i in range(8)]):
    a.imshow((x.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)); a.set_title(labels[y], fontsize=8); a.axis("off")
plt.show()

## Step 4: Training function
- Class weights: rare classes (loose_smut, stem_rust) count more in the loss, so the model does not ignore them.
- Mixed precision on GPU for speed, cosine learning-rate schedule.
- Keeps the epoch with the best **validation macro-F1** and stops early if it stops improving.
- **Pretrained weights:** taken from an attached `efficientnet_b0*.safetensors` file if present (dataset `efficientnet-b0-weights`), otherwise downloaded, which needs Internet on.

In [ ]:
def find_weights():
    """Any weights file in the attached inputs, whatever Kaggle named it."""
    files = [p for ext in (".safetensors", ".pth", ".pt", ".bin") for p in find_inputs(f"*{ext}")]
    files = [p for p in files if p.stat().st_size > 1_000_000]          # skip tiny files
    named = [p for p in files if "efficientnet" in str(p).lower()]
    return sorted(named or files)

def make_model(num_classes):
    """EfficientNet-B0 with ImageNet weights. Uses an attached weights file if there is one,
    so the notebook also works with Internet off (attach the efficientnet-b0-weights dataset)."""
    if not PRETRAINED:
        return timm.create_model(ARCH, pretrained=False, num_classes=num_classes)
    local = find_weights()
    if local:
        print("Pretrained weights from attached file:", local[0])
        return timm.create_model(ARCH, pretrained=True, num_classes=num_classes,
                                 pretrained_cfg_overlay=dict(file=str(local[0])))
    skip = {"datasets", "notebooks", "competitions", "models"}
    attached = sorted({Path(p).name for p in _glob.glob(str(INPUT_ROOT / "*" / "*" / "*"))}
                      | {p for p in os.listdir(INPUT_ROOT) if p not in skip})
    print("No weights file found in /kaggle/input. Attached:", attached)
    print("Attach the efficientnet-b0-weights dataset (Add Input), or turn Internet on. Trying to download...")
    return timm.create_model(ARCH, pretrained=True, num_classes=num_classes)

def run_epoch(model, dl, crit, opt=None, scaler=None):
    train = opt is not None
    model.train(train); total, ys, ps = 0.0, [], []
    for x, y in dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train), torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            out = model(x); loss = crit(out, y)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item() * len(y); ys += y.tolist(); ps += out.argmax(1).tolist()
    return total / len(dl.dataset), accuracy_score(ys, ps), f1_score(ys, ps, average="macro")

def train_task(name):
    d, labels = TASKS[name]
    tr, va, te = loaders(d)
    counts = np.bincount(d[d.split == "train"].y, minlength=len(labels))
    w = (1.0 / np.maximum(counts, 1)) ** 0.5
    w = torch.tensor(w / w.sum() * len(w), dtype=torch.float32)
    crit = nn.CrossEntropyLoss(weight=w.to(DEVICE), label_smoothing=0.1)

    model = make_model(len(labels)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS[name])
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

    hist, best, best_state, wait = [], -1, None, 0
    for ep in range(1, EPOCHS[name] + 1):
        t = time.time()
        tl, ta, tf1 = run_epoch(model, tr, crit, opt, scaler)
        vl, vacc, vf1 = run_epoch(model, va, crit)
        sched.step()
        hist.append(dict(epoch=ep, train_loss=round(tl, 4), val_loss=round(vl, 4), train_acc=round(ta, 4),
                         val_acc=round(vacc, 4), val_macro_f1=round(vf1, 4)))
        print(f"[{name}] ep {ep:2d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {vacc:.3f} F1 {vf1:.3f} | {time.time()-t:.0f}s")
        if vf1 > best: best, best_state, wait = vf1, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE: print("early stop"); break
    model.load_state_dict(best_state)
    torch.save(best_state, OUT / f"{name}.pt")
    return model, hist, te, labels

## Step 5: Train the health model
About 10 to 20 minutes on a T4 GPU.

In [ ]:
health_model, health_hist, health_test, health_labels = train_task("health")

## Step 6: Train the stage model
Small dataset (under 500 images), so this is quick but its test score is less reliable. Treat it as a first version.

In [ ]:
stage_model, stage_hist, stage_test, stage_labels = train_task("stage")

## Step 7: Evaluate on the test set
The test set was never used for training or for picking the best epoch, so these are the honest numbers.
Shows per-class precision / recall / F1 and a confusion matrix for each model.
Also scores the same test images as cheap-phone copies (low resolution, heavy JPEG), so you can see how much accuracy drops on poor photos.

In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval(); ys, ps = [], []
    for x, y in dl:
        ps += model(x.to(DEVICE)).argmax(1).tolist(); ys += y.tolist()
    return np.array(ys), np.array(ps)

def evaluate(name, model, dl, labels):
    y, p = predict(model, dl)
    idx = list(range(len(labels)))
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=idx, zero_division=0)
    cm = confusion_matrix(y, p, labels=idx)
    res = {
        "accuracy": round(float(accuracy_score(y, p)), 4),
        "macro_f1": round(float(f1_score(y, p, labels=idx, average="macro", zero_division=0)), 4),
        "test_images": int(len(y)),
        "labels": labels,
        "per_class": [dict(label=l, precision=round(float(a), 4), recall=round(float(b), 4), f1=round(float(c), 4), support=int(s))
                      for l, a, b, c, s in zip(labels, pr, rc, f1, sup)],
        "confusion_matrix": cm.tolist(),
    }
    print(f"\n== {name}: accuracy {res['accuracy']:.3f} | macro-F1 {res['macro_f1']:.3f} | {len(y)} test images")
    print(pd.DataFrame(res["per_class"]).to_string(index=False))
    fig, ax = plt.subplots(figsize=(1 + 0.7 * len(labels), 0.8 + 0.6 * len(labels)))
    ax.imshow(cm, cmap="YlOrBr")
    ax.set_xticks(idx, labels, rotation=45, ha="right"); ax.set_yticks(idx, labels)
    for i in idx:
        for j in idx: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{name} confusion matrix (test)")
    plt.tight_layout(); plt.show()
    return res

health_res = evaluate("health", health_model, health_test, health_labels)
stage_res = evaluate("stage", stage_model, stage_test, stage_labels)

# Same test images as a cheap phone would take them
def lowq_accuracy(model, d, labels):
    te = d[d.split == "test"]
    dl = DataLoader(ImgDS(te, lowq_tf), batch_size=BATCH, num_workers=0)
    y, p = predict(model, dl)
    return round(float(accuracy_score(y, p)), 4)

health_res["low_quality_accuracy"] = lowq_accuracy(health_model, TASKS["health"][0], health_labels)
stage_res["low_quality_accuracy"] = lowq_accuracy(stage_model, TASKS["stage"][0], stage_labels)
print("Cheap-phone test accuracy | health:", health_res["low_quality_accuracy"], "| stage:", stage_res["low_quality_accuracy"])

## Step 7b: Healthy-ear check
The test score above is mostly leaf photos, so it can look good while the model still calls every healthy
ear or field photo `loose_smut` (the v0.1 bug). This scores only the healthy ear and field photos in the health
test set. Aim for under 5% called a disease before you deploy.

In [ ]:
EAR_SOURCES = ["wheatphenology", "gwhd-stages", "wheat-weed-field-image-dataset", "global-wheat-detection", "global-wheat-head-dataset-2021", "gwhd-2021"]
d, _ = TASKS["health"]
ears = d[(d.split == "test") & (d.label == "healthy") & d.source.isin(EAR_SOURCES)]
if len(ears) == 0:
    print("No healthy ear photos in the health test set. Re-run Notebook 1 (ALSO_HEALTHY).")
else:
    y, p = predict(health_model, DataLoader(ImgDS(ears, eval_tf), batch_size=BATCH, num_workers=0))
    called = pd.Series([health_labels[i] for i in p]).value_counts()
    smut = float((called.get("loose_smut", 0)) / len(ears))
    disease = float(1 - called.get("healthy", 0) / len(ears))
    print(f"Healthy ear/field test photos: {len(ears)} from {sorted(ears.source.unique())}")
    print(f"  called loose_smut: {smut:.1%} | called any disease or not_wheat: {disease:.1%}")
    print(called.to_string())
    print("PASS" if disease < 0.05 else "CHECK: too many healthy ears called diseased; add more healthy ear photos")

## Step 8: Training curves
Validation macro-F1 and loss per epoch. Train and validation lines far apart = overfitting.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for name, h in [("health", health_hist), ("stage", stage_hist)]:
    h = pd.DataFrame(h)
    ax[0].plot(h.epoch, h.val_macro_f1, label=f"{name} val F1")
    ax[1].plot(h.epoch, h.train_loss, "--", label=f"{name} train"); ax[1].plot(h.epoch, h.val_loss, label=f"{name} val")
ax[0].set_title("Validation macro-F1"); ax[1].set_title("Loss"); ax[0].legend(); ax[1].legend()
plt.show()

## Step 9: Export one ONNX file for the API
Packs both models into `fasl_wheat.onnx`:
- input `image`: float32 `[N, 3, 224, 224]`, padded to square, ImageNet-normalised
- outputs `stage` and `health`: raw scores (logits)

Then checks the ONNX output matches PyTorch.

With Internet off the export is skipped. Then download `health.pt`, `stage.pt` and `labels.json` and run `python ml/scripts/export_onnx.py <folder>` on your PC; it builds and checks `fasl_wheat.onnx`.

In [ ]:
# Needs Internet. Without it the export below is skipped and you export on your own PC instead
# (download health.pt and stage.pt, then run ml/scripts/export_onnx.py).
!pip install -q onnx onnxruntime || echo "pip failed (Internet off?) - ONNX export will be skipped"

In [ ]:
class Combined(nn.Module):
    def __init__(self, stage, health):
        super().__init__(); self.stage, self.health = stage, health
    def forward(self, image): return self.stage(image), self.health(image)

combo = Combined(copy.deepcopy(stage_model).float().cpu(), copy.deepcopy(health_model).float().cpu()).eval()
onnx_path = OUT / "fasl_wheat.onnx"
infer_ms = None
try:
    torch.onnx.export(combo, torch.randn(1, 3, IMG, IMG), onnx_path, input_names=["image"], output_names=["stage", "health"],
                      dynamic_axes={"image": {0: "n"}, "stage": {0: "n"}, "health": {0: "n"}}, opset_version=17, dynamo=False)
    print("ONNX size:", round(onnx_path.stat().st_size / 1e6, 1), "MB")
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    x = torch.randn(4, 3, IMG, IMG)
    with torch.no_grad(): ts, th = combo(x)
    os_, oh = sess.run(["stage", "health"], {"image": x.numpy()})
    print("max diff stage:", float(np.abs(ts.numpy() - os_).max()), "| health:", float(np.abs(th.numpy() - oh).max()))
    t = time.time()
    for _ in range(20): sess.run(None, {"image": x[:1].numpy()})
    infer_ms = round((time.time() - t) / 20 * 1000, 1)
    print("CPU inference per image:", infer_ms, "ms")
except Exception as e:
    print("ONNX export/check skipped:", type(e).__name__, e)
    print("Not a problem: health.pt and stage.pt are saved. Download them and run ml/scripts/export_onnx.py on your PC.")

## Step 10: Write `metrics.json` and `labels.json`
- `metrics.json` goes to the website (`public/metrics.json`). Every number in it comes from this run.
- `labels.json` goes to the API (`ml/config/labels.json`) so class order matches the model.

In [ ]:
counts = lambda name: {s: int((TASKS[name][0].split == s).sum()) for s in ["train", "val", "test"]}
metrics = {
    "model_version": MODEL_VERSION,
    "architecture": ARCH,
    "image_size": IMG,
    "dataset": {"health": counts("health"), "stage": counts("stage"),
                "sources": sorted(man.source.unique().tolist())},
    "heads": {"health": health_res, "stage": stage_res},
    "training_curve": {"health": health_hist, "stage": stage_hist},
    "inference_ms_cpu": infer_ms,
}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
labels_cfg = {"crop": "wheat", "model_version": MODEL_VERSION, "image_size": IMG, "mean": MEAN, "std": STD,
              "growth_stage": stage_labels, "health": health_labels, "confidence_threshold": 0.6}
(OUT / "labels.json").write_text(json.dumps(labels_cfg, indent=2))
print(json.dumps({k: metrics["heads"][k]["accuracy"] for k in metrics["heads"]}, indent=2))
print("Saved:", sorted(p.name for p in OUT.iterdir()))

## Step 11: Next
1. **Save Version → Save & Run All (Commit)** with GPU on.
2. Open the version → **Output** → download `fasl_wheat.onnx`, `metrics.json`, `labels.json`.
3. Send me the numbers printed in Step 7, and we will decide what to improve (more stage data, more epochs, a bigger model).